In [ ]:
import torch
import os
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch.utils.data
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil
import torchvision
from torchvision.transforms import (
    Compose,
    Lambda,
    RandomCrop,
    RandomHorizontalFlip,
    CenterCrop
)
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import cv2
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import mlflow.pytorch
from mlflow import MlflowClient
import optuna
import mlflow

In [ ]:
# TODO here dataloading
# TODO train test split
# TODO data visualization
# TODO further analysis if required

In [ ]:
#Dataset class

In [ ]:
class MyDataSet(Dataset):
    def __init__(self, file_path, label_map, augment):
        self.df = pd.read_csv(file_path)
        self.label_map = label_map
        self.output_frames = 30
        self.height = self.width = 128
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        video_path, label = row['path'], row['label']
        flow_tensor = self.process(video_path=video_path)
        return flow_tensor, torch.tensor(self.label_map[label], dtype=torch.long)

    def process(self, video_path: str):
        video = cv2.VideoCapture(filename=video_path)
        total = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        # output_frames+1 raw frames -> output_frames flow pairs
        raw_frames = np.linspace(0, total - 1, self.output_frames + 1, dtype=int)

        final_frames = []
        for frame_number in raw_frames:
            video.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
            res, frame = video.read()
            if not res:
                frame = final_frames[-1] if final_frames else np.zeros(
                    (self.height, self.width, 3), dtype=np.uint8)
                final_frames.append(frame)
                continue
            final_frames.append(cv2.resize(frame, (self.width, self.height)))
        video.release()

        flow_frames = []
        for i in range(self.output_frames):
            # Convert to gray scale
            prvs = cv2.cvtColor(final_frames[i], cv2.COLOR_BGR2GRAY)
            nxt_frame = cv2.cvtColor(final_frames[i + 1], cv2.COLOR_BGR2GRAY)
            flow = cv2.calcOpticalFlowFarneback(prvs, nxt_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)  #type:ignore
            fx, fy = flow[..., 0], flow[..., 1]
            mag, ang = cv2.cartToPolar(fx, fy)
            scale = mag.max() + 1e-6

            # Encode as 3 channels, all in [0, 1]
            ch = np.stack([
                np.clip(fx / scale * 0.5 + 0.5, 0, 1),  # horizontal flow
                np.clip(fy / scale * 0.5 + 0.5, 0, 1),  # vertical flow
                np.clip(mag / scale,             0, 1),  # magnitude
            ], axis=0).astype(np.float32)
            flow_frames.append(torch.from_numpy(ch))

        return torch.stack(flow_frames)  # (output_frames, 3, H, W)

In [ ]:
label_map = {
    "geste_0": 0,
    "geste_1" :1,
    "geste_2": 2,
    "class_1": 3,
    "class_2": 4,
    "Gesture01":5,
    "Gesture02":6,
    "hand_turn":7,
    "ok_sign":8,
    "thumb_up":9
}

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
class MyModel(nn.Module):
    """Per-frame ResNet encoder + LSTM over time for dynamic gesture clips.

    Input : (B, T, 3, H, W)  -- sequence of T optical-flow frames
    Output: (B, num_classes)
    """

    def __init__(self, num_classes, hidden_size=256, num_layers=1,
                 bidirectional=True, pretrained=True, freeze_cnn=False, dropout=0.5):
        super().__init__()

        # 1) CNN feature extractor: ResNet-18 with its classifier head removed
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)
        self.feat_dim = backbone.fc.in_features      # 512 for resnet18
        backbone.fc = nn.Identity()                  # -> outputs the 512-d feature
        self.cnn = backbone

        if freeze_cnn:                               # optional: use ResNet as a fixed encoder
            for p in self.cnn.parameters():
                p.requires_grad = False

        # 2) LSTM over the temporal sequence of per-frame features
        self.lstm = nn.LSTM(
            input_size=self.feat_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,   # PyTorch ignores dropout when num_layers==1
        )

        # 3) Classifier on the final temporal representation
        lstm_out = hidden_size * 2
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_out, num_classes),
        )

    def forward(self, x):                  # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)      # fold time into the batch dim
        feats = self.cnn(x)               # (B*T, feat_dim)
        feats = feats.view(B, T, -1)      # (B, T, feat_dim)

        out, _ = self.lstm(feats)         # (B, T, lstm_out)
        last = out[:, -1, :]              # representation at the final time step
        return self.classifier(last)      # (B, num_classes)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

device = "cuda" if torch.cuda.is_available() else "cpu"
GLOBAL_BEST_VAL_ACC = 0.0   # best val acc across ALL Optuna trials (for model saving)


def macro_prf(y_true, y_pred):
    """Macro precision, recall, F1 (in %) for a list of label tensors."""
    yt = torch.cat(y_true).numpy()
    yp = torch.cat(y_pred).numpy()
    p, r, f, _ = precision_recall_fscore_support(
        yt, yp, average="macro", zero_division=0)
    return p * 100, r * 100, f * 100


def new_train(lr=1e-3, hidden_size=128, num_layers=1, dropout=0.5,
              num_epochs=50, trial=None):
    """Train MyModel with the given hyperparameters.

    Logs params and per-epoch metrics (loss, accuracy, macro precision/recall/F1
    for both train and val) to the active MLflow run. Returns the best val
    accuracy reached. If an Optuna `trial` is passed, reports per-epoch val acc
    for pruning. Saves the GLOBALLY best model (across all trials) to "my_model.pth".
    """
    global GLOBAL_BEST_VAL_ACC

    # Log the hyperparameters of this run
    mlflow.log_params({
        "lr": lr,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "dropout": dropout,
        "num_epochs": num_epochs,
        "optimizer": "Adam",
    })

    model = MyModel(
        num_classes=10, hidden_size=hidden_size,
        num_layers=num_layers, dropout=dropout,
    ).to(device)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)
    loss_criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    train_loader = DataLoader(MyDataSet("train.csv", label_map=label_map, augment=True),  batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(MyDataSet("val.csv",   label_map=label_map, augment=False), batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_true, train_pred = [], []

        for videos, labels in train_loader:
            videos, labels = videos.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(videos)
            loss = loss_criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            train_loss    += loss.item() * videos.size(0)
            train_correct += (preds == labels).sum().item()
            train_total   += videos.size(0)
            train_true.append(labels.cpu()); train_pred.append(preds.detach().cpu())

        model.eval()
        val_correct, val_total = 0, 0
        val_true, val_pred = [], []                     # <-- add
        with torch.no_grad():
            for videos, labels in val_loader:
                videos, labels = videos.to(device), labels.to(device)
                outputs = model(videos)
                preds = outputs.argmax(1)               # <-- name it
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)
                val_true.append(labels.cpu())           # <-- add
                val_pred.append(preds.cpu())            # <-- add


        train_acc = train_correct / train_total * 100
        val_acc = val_correct / val_total * 100
        yt = torch.cat(val_true).numpy()
        yp = torch.cat(val_pred).numpy()
        val_p, val_r, val_f1, _ = precision_recall_fscore_support(
            yt, yp, average="macro", zero_division=0)   # macro = unweighted mean over classes

        mlflow.log_metric("val_precision", val_p * 100, step=epoch)
        mlflow.log_metric("val_recall",    val_r * 100, step=epoch)
        mlflow.log_metric("val_f1",        val_f1 * 100, step=epoch)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        # Save only when this beats the best across ALL trials so far, and store
        # the architecture config so the checkpoint can be reloaded standalone.
        if val_acc > GLOBAL_BEST_VAL_ACC:
            GLOBAL_BEST_VAL_ACC = val_acc
            torch.save({
                "state_dict":  model.state_dict(),
                "hidden_size": hidden_size,
                "num_layers":  num_layers,
                "dropout":     dropout,
                "val_acc":     val_acc,
            }, "my_model.pth")

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | train acc: {train_acc:.1f}%  "
                  f"val acc: {val_acc:.1f}%  val F1: {va_f1:.1f}%  (best acc: {best_val_acc:.1f}%)")

        # Optuna pruning: report intermediate value and stop unpromising trials early
        if trial is not None:
            trial.report(val_acc, epoch)
            if trial.should_prune():
                mlflow.log_metric("best_val_acc", best_val_acc)
                raise optuna.TrialPruned()

    mlflow.log_metric("best_val_acc", best_val_acc)
    return best_val_acc

In [ ]:
mlflow.set_experiment("handgesten_final")

N_TRIALS   = 1    # number of hyperparameter combinations to try
NUM_EPOCHS = 2    # epochs per trial (pruning stops weak trials early)


def objective(trial):
    # Search space
    lr          = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])
    num_layers  = trial.suggest_int("num_layers", 1, 3)
    dropout     = trial.suggest_float("dropout", 0.1, 0.6)

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        best_val_acc = new_train(
            lr=lr, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout, num_epochs=NUM_EPOCHS, trial=trial,
        )
    return best_val_acc


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)

GLOBAL_BEST_VAL_ACC = 0.0   # reset best-model tracker for a fresh sweep

with mlflow.start_run(run_name="my_run"):
    study.optimize(objective, n_trials=N_TRIALS)

    # Record the best configuration on the parent run
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_acc", study.best_value)

print("Best val acc:", study.best_value)
print("Best params :", study.best_params)